# Grokking Diffusion: Tiny MNIST Generator

**Goal:** Train the smallest possible diffusion model that can generate recognizable MNIST digits.

**Why diffusion?** Unlike GANs, diffusion models are stable to train and scale down gracefully.

**Target:**
- <10,000 parameters
- Recognizable digit generation
- Class-conditional generation (optional)

**Architecture:** Minimal U-Net with attention, or a tiny transformer denoiser.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Loading

In [ ]:
# Simple transforms - normalize to [-1, 1] for diffusion
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Map to [-1, 1]
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

print(f"Training samples: {len(train_dataset)}")

## 3. Diffusion Utilities

Simple DDPM-style diffusion with:
- Linear noise schedule
- Predicting noise (ε-prediction)

In [ ]:
class SimpleDiffusion:
    """
    Minimal DDPM implementation.
    """
    
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.num_timesteps = num_timesteps
        
        # Linear schedule
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        
        # For sampling
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - self.alpha_bars)
        
    def add_noise(self, x, t, noise=None):
        """
        Add noise to x at timestep t.
        q(x_t | x_0) = N(sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)
        """
        if noise is None:
            noise = torch.randn_like(x)
        
        sqrt_alpha_bar = self.sqrt_alpha_bars[t].view(-1, 1, 1, 1).to(x.device)
        sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alpha_bars[t].view(-1, 1, 1, 1).to(x.device)
        
        return sqrt_alpha_bar * x + sqrt_one_minus_alpha_bar * noise, noise
    
    @torch.no_grad()
    def sample(self, model, shape, device):
        """
        Generate samples via DDPM sampling.
        """
        model.eval()
        
        # Start from pure noise
        x = torch.randn(shape, device=device)
        
        for t in tqdm(reversed(range(self.num_timesteps)), desc='Sampling'):
            t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
            
            # Predict noise
            pred_noise = model(x, t_batch)
            
            # Get parameters
            alpha = self.alphas[t].to(device)
            alpha_bar = self.alpha_bars[t].to(device)
            beta = self.betas[t].to(device)
            
            # Denoise
            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = 0
            
            # x_{t-1} = (x_t - beta_t / sqrt(1 - alpha_bar_t) * eps) / sqrt(alpha_t) + sqrt(beta_t) * z
            x = (x - beta / torch.sqrt(1 - alpha_bar) * pred_noise) / torch.sqrt(alpha)
            x = x + torch.sqrt(beta) * noise
        
        return x


diffusion = SimpleDiffusion(num_timesteps=1000)
print(f"Diffusion setup: {diffusion.num_timesteps} timesteps")

## 4. Tiny Denoiser Architecture

**Design for minimal parameters:**
- Small number of channels
- Minimal downsampling (or none)
- Time embedding via sinusoidal + small MLP
- Simple residual blocks

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal time embeddings."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class TinyResBlock(nn.Module):
    """Minimal residual block with time conditioning."""
    
    def __init__(self, channels, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, channels)
        self.norm1 = nn.GroupNorm(4, channels)
        self.norm2 = nn.GroupNorm(4, channels)
        
    def forward(self, x, t_emb):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        
        # Add time embedding
        t = self.time_mlp(t_emb)[:, :, None, None]
        h = h + t
        
        h = self.norm2(h)
        h = F.silu(h)
        h = self.conv2(h)
        
        return x + h


class TinyDiffusionModel(nn.Module):
    """
    Minimal diffusion denoiser for MNIST.
    
    Target: <10,000 parameters
    """
    
    def __init__(self, channels=16, time_dim=32):
        super().__init__()
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        
        # Input projection
        self.input_conv = nn.Conv2d(1, channels, 3, padding=1)
        
        # Residual blocks (no downsampling to save params)
        self.res1 = TinyResBlock(channels, time_dim)
        self.res2 = TinyResBlock(channels, time_dim)
        
        # Output projection
        self.output_conv = nn.Sequential(
            nn.GroupNorm(4, channels),
            nn.SiLU(),
            nn.Conv2d(channels, 1, 3, padding=1)
        )
        
    def forward(self, x, t):
        """
        x: (B, 1, 28, 28) noisy image
        t: (B,) timesteps
        Returns: (B, 1, 28, 28) predicted noise
        """
        # Time embedding
        t_emb = self.time_embed(t.float())
        
        # Process
        h = self.input_conv(x)
        h = self.res1(h, t_emb)
        h = self.res2(h, t_emb)
        h = self.output_conv(h)
        
        return h


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Test model
model = TinyDiffusionModel(channels=16, time_dim=32).to(device)
total_params = count_parameters(model)

print(f"\nParameter count: {total_params:,}")
print(f"Target: <10,000")
print(f"Status: {'PASS ✅' if total_params < 10000 else 'OVER BUDGET ❌'}")

# Test forward pass
x = torch.randn(2, 1, 28, 28, device=device)
t = torch.randint(0, 1000, (2,), device=device)
out = model(x, t)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {out.shape}")

## 5. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, diffusion, device):
    model.train()
    total_loss = 0
    
    for images, _ in loader:
        images = images.to(device)
        batch_size = images.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, diffusion.num_timesteps, (batch_size,), device=device)
        
        # Add noise
        noise = torch.randn_like(images)
        noisy_images, _ = diffusion.add_noise(images, t, noise)
        
        # Predict noise
        optimizer.zero_grad()
        pred_noise = model(noisy_images, t)
        
        # MSE loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_size
    
    return total_loss / len(loader.dataset)

In [ ]:
# Training configuration
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01

# Create model
model = TinyDiffusionModel(channels=16, time_dim=32).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Model parameters: {count_parameters(model):,}")
print(f"Training for {NUM_EPOCHS} epochs...")
print("="*50)

In [ ]:
# Training
history = {'loss': []}
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, diffusion, device)
    scheduler.step()
    history['loss'].append(loss)
    
    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), 'best_diffusion_model.pth')
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {loss:.6f} | Best: {best_loss:.6f}")

print("="*50)
print(f"Training complete! Best loss: {best_loss:.6f}")

## 6. Generate Samples

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_diffusion_model.pth'))

# Generate samples
print("Generating samples...")
samples = diffusion.sample(model, (16, 1, 28, 28), device)

# Denormalize
samples = (samples + 1) / 2  # [-1, 1] -> [0, 1]
samples = samples.clamp(0, 1)

# Plot
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i, 0].cpu().numpy(), cmap='gray')
    ax.axis('off')

plt.suptitle(f'Generated MNIST ({count_parameters(model):,} params)', fontsize=14)
plt.tight_layout()
plt.savefig('generated_samples.png', dpi=150)
plt.show()

## 7. Compare with Real Digits

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

# Real samples
real_batch = next(iter(train_loader))[0][:8]
real_batch = (real_batch + 1) / 2  # Denormalize

for i in range(8):
    axes[0, i].imshow(real_batch[i, 0].cpu().numpy(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Real', fontsize=12)
    
    axes[1, i].imshow(samples[i, 0].cpu().numpy(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Generated', fontsize=12)

plt.suptitle('Real vs Generated MNIST', fontsize=14)
plt.tight_layout()
plt.savefig('real_vs_generated.png', dpi=150)
plt.show()

## 8. Summary

In [ ]:
print("="*60)
print("GROKKING DIFFUSION EXPERIMENT SUMMARY")
print("="*60)
print(f"\n📊 Model: Tiny Diffusion")
print(f"   - Parameters: {count_parameters(model):,}")
print(f"   - Channels: 16")
print(f"   - Time embedding dim: 32")
print(f"   - ResBlocks: 2")
print(f"\n🎨 Generation:")
print(f"   - Diffusion steps: {diffusion.num_timesteps}")
print(f"   - Image size: 28×28")
print(f"\n📈 Training:")
print(f"   - Epochs: {NUM_EPOCHS}")
print(f"   - Best loss: {best_loss:.6f}")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"\n💡 Results:")
print(f"   - Target: <10,000 params")
print(f"   - Actual: {count_parameters(model):,} params")
print(f"   - Status: {'ACHIEVED ✅' if count_parameters(model) < 10000 else 'OVER BUDGET ❌'}")
print("="*60)